---
---
---

# Data

---
---
---

<br>

> **_Abstract:_** The notebook focuses on data preparation for the weather nowcasting project. The dataset contains **_physical measurements from 19 meteorological stations_** in the Lake Como area with a temporal resolution of **_5 minutes_**, which are used to predict the weather conditions in the next 30-60-90-120 minutes.

> **_Table of Contents:_**
>
> 1. DataFrame
> 2. Feature Engineering
>     - Days, Hours, Minutes (Data Collection)
>     - Time Features
>     - Feature Selection
> 3. Exploratory Data Analysis (EDA)
>     - Profiling
>     - Correlations
>     - Target Analysis
>     - Outliers

<br>

> | **_Language:_** python@3.12.10 |
> | - |

<br>

> | **_Source:_** "notebook/data.ipynb" |
> | - |

<br>

> | **_Configurations:_** "config" |
> | - |

<br>

> | **_Libraries:_** "lib" |
> | - |

<br>

---
---
---

## Dependencies

### Packages

In [ ]:
from google.cloud import bigquery
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sys
import warnings

ROOT_PATH = Path.cwd().resolve()
if ROOT_PATH.name == "notebook":
    ROOT_PATH = ROOT_PATH.parent
if str(ROOT_PATH) not in sys.path:
    sys.path.insert(0, str(ROOT_PATH))

from lib import config as the_config
from lib import utils as the_utils
from lib import data_profiling as profiler_lib

# -------------------------

%matplotlib inline
the_config.refresh_logging()
warnings.filterwarnings("ignore")

### Constants

In [ ]:
# Feature Engineering
ML_TARGET = "__"

# EDA
CORR_THR = 0.25

# GCP
BQ_CLIENT = bigquery.Client(project=the_config.GCP_PROJECT)

### Paths

In [ ]:
for path in the_config.PATHS:
    the_utils.ensure_path(path)

## DataFrame

> **_Source:_** "larionow_dataset.measurements" table on Google BigQuery.

> The dataset contains **_physical measurements from 19 meteorological stations_** in the Lake Como area with a temporal resolution of **_5 minutes_**.

In [ ]:
query = f"""
SELECT * FROM `{the_config.BQ_TABLE}`;
"""
df = BQ_CLIENT.query(query).to_dataframe()
display(df.head())
display(df.tail())

## Feature Engineering

### Time Features

> To model the **_time series data_**, we need to create specific time features. We will retrieve the **_following features:_**
>
> - **_Date_**: The date of the observation.
> - **_Year_**: The year of the observation.
> - **_Month_**: The month of the observation.
> - **_Day_**: The day of the month of the observation.
> - **_Hour_**: The hour of the observation.
> - **_Minute_**: The minute of the observation.
> - **_Quarter_**: The quarter of the observation.
> - **_Week of Year_**: The week of the year of the observation.
> - **_Day of Year_**: The day of the year of the observation.
> - **_Day of Week_**: The day of the week of the observation.

### Days

> The dataset is expected to contain **5472 rows per day**, based on the **_following calculation:_**
>
> $$
> 19\ \text{stations} \times 12\ \text{measurements/hour} \times 24\ \text{hours} = 5472\ \text{rows/day}
> $$

In [ ]:
df["day"].value_counts()

### Hours

> The dataset is expected to contain **_228 rows per hour_**, based on the **_following calculation:_**
>
> $$
> 19 \times 12 = 228\ \text{rows/hour}
> $$
>
> Hence, the number of days represented for a given hour of the day is **_computed as:_**
>
> $$
> \frac{N}{228} = \text{days collected for that hour}
> $$
>
> Where $N$ is the **_total number of rows collected_** for that hour across the entire dataset.

In [ ]:
df["hour"].sort_values().value_counts()

### Minutes

> The dataset is expected to contain **_456 rows per minute (attribute)_**, based on the **_following calculation:_**
>
> $$
> 19 \times 24 = 456\ \text{rows/minute}
> $$
>
> Hence, the number of days represented for a given minute value is **_computed as:_**
>
> $$
> \frac{N}{456} = \text{days collected for that minute}
> $$
>
> Where $N$ is the **_total number of rows collected_** for that minute across the entire dataset.

In [ ]:
df["minute"].sort_values().value_counts()

### Feature Selection

> Based on the **_analyses performed for the task_**, the following features are selected for training from the overall data.

> Temporal:
> - **_Time Features_**
>
> Categorical:
> - **_Locations:_** `station`, `city`, `province`
> - **_Measurements:_** `wind_dir`
> 
> Numerical:
> - **_Location-based:_** `latitude`, `longitude`, `altitude_m`
> - **_Measurements:_** `temperature_c`, `humidity_pct`, `dew_point_c`, `wind_speed_kmh`, `pressure_hpa`, `rain_mm`, `rain_mmh`
> - **_Metrics:_** conf_`temperature_c`, `conf_humidity_pct`, `conf_dew_point_c`, `conf_wind_speed_kmh`, `conf_pressure_hpa`, `conf_rain_mm`, `conf_rain_mmh`, `conf_overall_gmean`
>
>
> Target:
> - **_???_**

In [ ]:
df.info()

## Exploratory Data Analysis (EDA)

### Profiling

> Profiling data helps us to **_understand the distribution_** of the data and detect potential problems.

In [ ]:
profiler_lib.profile_data(
    df,
    out_path=(the_config.PROFILING_PATH),
    title="Profiling"
)

### Correlations

> ...

In [ ]:
corr_full = df.select_dtypes(include="number").corr()
corr_thr = corr_full.copy()
corr_thr[np.abs(corr_thr) <= CORR_THR] = 0
mask = np.triu(np.ones_like(corr_full, dtype=bool), k=1)
fig = plt.figure(figsize=(20, 8))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.05], wspace=0.15)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
cax = fig.add_subplot(gs[0, 2])

sns.heatmap(
    corr_full,
    mask=mask,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.7,
    linecolor="lightgray",
    cbar=False,
    ax=ax1
)

sns.heatmap(
    corr_thr,
    mask=mask,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.7,
    linecolor="lightgray",
    cbar=True,
    cbar_ax=cax,
    ax=ax2
)

for ax in (ax1, ax2):
    ax.set_xticks(np.arange(len(corr_full.columns)) + 0.5)
    ax.set_xticklabels(corr_full.columns, rotation=90)
    ax.set_yticks(np.arange(len(corr_full.index)) + 0.5)
    ax.set_yticklabels(corr_full.index)

fig.suptitle("Correlation Matrix", fontsize=18)
ax1.set_title("FULL", pad=16)
ax2.set_title(r"$|\rho| > 0.25$", pad=16)

plt.show()

### Target Analysis

> ...

In [ ]:
# df[ML_TARGET].describe()

> ...

In [ ]:
# n_bins = len(df) // 100
# df[ML_TARGET].plot.hist(bins=n_bins)

### Outliers

> ...

In [ ]:
# # IQR Method
# Q1 = df[ML_TARGET].quantile(0.25)
# Q3 = df[ML_TARGET].quantile(0.75)
# IQR = Q3 - Q1
# lower = Q1 - 1.5 * IQR
# upper = Q3 + 1.5 * IQR
# mask_outlier = (df[ML_TARGET] < lower) | (df[ML_TARGET] > upper)
# outliers = df[mask_outlier]
# logger.debug(f"Outliers (count): {len(outliers)} / {len(df)}")
# logger.debug(f"{"":3}Outliers (%): {len(outliers) / len(df) * 100:.2f}%")

# # NOT (y < lower OR y > upper)
# display(df[ML_TARGET][~mask_outlier].describe())

# # IS (y < lower OR y > upper)
# display(df[ML_TARGET][mask_outlier].describe())

> The way to handle outliers depends on the **_model used_**. Using a **_tree-based model_** in generally more robust to outliers, while **_linear models_** can be significantly affected by them.

In [ ]:
# plt.hist(
#     df.loc[~mask_outlier, ML_TARGET],
#     bins=n_bins,
#     label="Inlier"
# )
# plt.hist(
#     df.loc[mask_outlier, ML_TARGET],
#     bins=n_bins,
#     alpha=0.7,
#     label="Outlier"
# )
# plt.axvline(lower, color="red", linestyle="--")
# plt.axvline(upper, color="red", linestyle="--")

# plt.title("Outlier Detection with IQR Method")
# plt.xlabel("TARGET NAME")
# plt.ylabel("Frequency")
# plt.legend()